# Testing Model Manual

---

In [1]:
import random
import numpy as np
import pandas as pd
import xgboost as xgb

In [2]:
MODEL_PATH = "../data/fraud_model.json"
THRESHOLD  = 0.9275

FEATURES = [
    "amount", "hour_of_day", "is_home", "is_domestic",
    "avg_amount_24h", "amount_ratio",
    "tx_count_1h", "tx_count_3h", "time_since_last_tx_sec",
]

model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)
print(f"Model loaded — {len(FEATURES)} features, threshold={THRESHOLD}")

Model loaded — 9 features, threshold=0.9275


In [88]:
# Khởi tạo df rỗng — reset cell này để bắt đầu lại từ đầu
df = pd.DataFrame(columns=FEATURES)
print("df initialized:", df.shape)

df initialized: (0, 9)


In [107]:
def make_tx(
    min_amount: float,
    max_amount: float,
    hour: int,
    *,
    is_home: bool = True,
    is_domestic: bool = True,
    avg_amount_24h: float = 5_000_000,
    tx_count_1h: int = 3,
    tx_count_3h: int = 3,
    time_since_last_tx_sec: float = 3_600,
) -> pd.DataFrame:
    amount = random.uniform(min_amount, max_amount)
    ratio  = amount / avg_amount_24h if avg_amount_24h > 0 else 0.0

    return pd.DataFrame([{
        "amount":                 round(amount, -3),
        "hour_of_day":            hour,
        "is_home":                int(is_home),
        "is_domestic":            int(is_domestic),
        "avg_amount_24h":         avg_amount_24h,
        "amount_ratio":           round(ratio, 6),
        "tx_count_1h":            tx_count_1h,
        "tx_count_3h":            tx_count_3h,
        "time_since_last_tx_sec": time_since_last_tx_sec,
    }])

In [108]:
# Gọi make_tx() và cộng dồn vào df
df_row = make_tx(
    min_amount=100_000_000,
    max_amount=150_000_000,
    hour=17
)
df = pd.concat([df, df_row], ignore_index=True)
df

,amount,hour_of_day,is_home,is_domestic,avg_amount_24h,amount_ratio,tx_count_1h,tx_count_3h,time_since_last_tx_sec
0,8741000.0,16,1,1,5000000,1.74816,0,0,3600
1,118900000.0,16,1,1,5000000,23.780045,1,1,3600
2,107014000.0,17,1,1,5000000,21.402894,2,2,3600
3,142295000.0,17,1,1,5000000,28.458945,3,3,3600


In [109]:
X    = df[FEATURES].astype(float)
dmat = xgb.DMatrix(X, feature_names=FEATURES)
risk_scores = model.get_booster().predict(dmat)

result = df[FEATURES].copy()
result["risk_score"] = risk_scores
result["is_fraud"]   = risk_scores >= THRESHOLD

result[["amount", "hour_of_day", "is_home", "is_domestic",
        "amount_ratio", "tx_count_1h", "tx_count_3h",
        "risk_score", "is_fraud"]]

,amount,hour_of_day,is_home,is_domestic,amount_ratio,tx_count_1h,tx_count_3h,risk_score,is_fraud
0,8741000.0,16,1,1,1.74816,0,0,0.000261,False
1,118900000.0,16,1,1,23.780045,1,1,0.388414,False
2,107014000.0,17,1,1,21.402894,2,2,0.100697,False
3,142295000.0,17,1,1,28.458945,3,3,0.992598,True


---
## Predict từ predict.csv

In [15]:
import pandas as pd
import xgboost as xgb
from datetime import timedelta

DOMESTIC_LOCS = {"Hanoi", "HCM City", "Da Nang", "Can Tho", "Hai Phong"}

MODEL_PATH = "../data/fraud_detection_model.json"
CSV_PATH   = "../data/predict.csv"
THRESHOLD  = 0.9688

FEATURES = [
    "amount", "hour_of_day", "is_home", "is_domestic",
    "avg_amount_24h", "amount_ratio",
    "tx_count_1h", "tx_count_3h", "time_since_last_tx_sec",
]

model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)

# Load và sort theo account + event_time
raw = pd.read_csv(CSV_PATH)
raw["event_time"] = pd.to_datetime(raw["event_time"])
raw = raw.sort_values(["account_id", "event_time"]).reset_index(drop=True)

# Derive home_location per account = location đầu tiên xuất hiện
home_map = raw.groupby("account_id")["location_name"].first().to_dict()

rows = []
for acc_id, grp in raw.groupby("account_id", sort=False):
    home = home_map[acc_id]
    history = []  # list of (event_time, amount)

    for _, tx in grp.iterrows():
        t      = tx["event_time"]
        amount = float(tx["amount"])
        loc    = tx["location_name"]

        # Features từ history (các tx TRƯỚC tx này)
        cutoff_1h  = t - timedelta(hours=1)
        cutoff_3h  = t - timedelta(hours=3)
        cutoff_24h = t - timedelta(hours=24)

        prev_24h = [(pt, pa) for pt, pa in history if pt >= cutoff_24h]
        avg_24h  = sum(pa for _, pa in prev_24h) / len(prev_24h) if prev_24h else amount
        cnt_1h   = sum(1 for pt, _ in history if pt >= cutoff_1h)
        cnt_3h   = sum(1 for pt, _ in history if pt >= cutoff_3h)
        last_ts  = max((pt for pt, _ in history), default=None)
        time_since = (t - last_ts).total_seconds() if last_ts else 0.0

        rows.append({
            "transaction_id":         tx["transaction_id"],
            "account_id":             acc_id,
            "event_time":             t,
            "location_name":          loc,
            "is_fraud_label":         tx["is_fraud"],
            "fraud_pattern":          tx["fraud_pattern"],
            "amount":                 amount,
            "hour_of_day":            t.hour,
            "is_home":                int(loc == home),
            "is_domestic":            int(loc in DOMESTIC_LOCS),
            "avg_amount_24h":         round(avg_24h, 2),
            "amount_ratio":           round(amount / avg_24h if avg_24h > 0 else 1.0, 6),
            "tx_count_1h":            cnt_1h,
            "tx_count_3h":            cnt_3h,
            "time_since_last_tx_sec": round(time_since, 1),
        })
        history.append((t, amount))

feat_df = pd.DataFrame(rows)

X    = feat_df[FEATURES].astype(float)
dmat = xgb.DMatrix(X, feature_names=FEATURES)
scores = model.get_booster().predict(dmat)

feat_df["risk_score"] = scores
feat_df["predicted"]  = scores >= THRESHOLD

cols = ["transaction_id", "account_id", "event_time", "location_name",
        "amount", "avg_amount_24h", "amount_ratio",
        "tx_count_1h", "tx_count_3h", "time_since_last_tx_sec",
        "is_home", "is_domestic", "hour_of_day",
        "risk_score", "predicted", "is_fraud_label", "fraud_pattern"]

feat_df[cols]

,transaction_id,account_id,event_time,location_name,amount,avg_amount_24h,amount_ratio,tx_count_1h,tx_count_3h,time_since_last_tx_sec,is_home,is_domestic,hour_of_day,risk_score,predicted,is_fraud_label,fraud_pattern
0,TEST_TX_001,TEST_ACC_001,2026-06-25 08:14:23,Hanoi,55000000.0,55000000.00,1.000000,0,0,0.0,1,1,8,0.000035,False,False,NaN
1,TEST_TX_002,TEST_ACC_001,2026-06-25 09:47:51,Hanoi,72000000.0,55000000.00,1.309091,0,1,5608.0,1,1,9,0.000981,False,False,NaN
2,TEST_TX_003,TEST_ACC_001,2026-06-25 11:03:09,Hanoi,63000000.0,63500000.00,0.992126,0,2,4518.0,1,1,11,0.000194,False,False,NaN
3,TEST_TX_004,TEST_ACC_001,2026-06-25 13:29:44,Hanoi,88000000.0,63333333.33,1.389474,0,1,8795.0,1,1,13,0.290279,False,False,NaN
4,TEST_TX_005,TEST_ACC_001,2026-06-25 15:55:17,Hanoi,51000000.0,69500000.00,0.733813,0,1,8733.0,1,1,15,0.000030,False,False,NaN
